# FusionClassifier — Evaluation Pack
This notebook evaluates a trained checkpoint and renders all metrics, plots, and error-analysis tables inline.

**Parameters are injected by `train.py` via `papermill` when called automatically, or you can set them manually in the `Parameters` cell below.**

In [ ]:
#Papermill parameters (overridden at execution time)
checkpoint_path: str = "checkpoints/best.pt"   # path to the .pt checkpoint
test_path:       str = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Test"
batch_size:      int = 32
num_workers:     int = 4
text_model_name: str = "distilbert-base-uncased"
image_encoder_name: str = "convnext_tiny"
text_encoder_name:  str = "distilbert"
dropout:         float = 0.2
max_length:      int = 32
local_files_only: bool = False
# If non-empty, class_names overrides the list read from the checkpoint
class_names_override: list = []

## 1. Imports & Helpers

In [ ]:
from __future__ import annotations

import importlib.util
import os
import sys
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
import pandas as pd

# Make src importable
notebook_dir = Path(globals().get('__vsc_ipynb_file__', Path.cwd() / 'dummy')).parent
# Fallback: assume notebook lives at src/evaluation/
src_dir = notebook_dir.parent if notebook_dir.name == 'evaluation' else Path(checkpoint_path).resolve().parent.parent / 'src'
if not src_dir.exists():
    # last resort: look two levels up from cwd
    src_dir = Path.cwd().parents[0] / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'src_dir: {src_dir}')

## 2. Load Checkpoint & Build Model

In [ ]:
from models.classifiers import FusionClassifier

ckpt_path = Path(checkpoint_path)
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'

ckpt = torch.load(ckpt_path, map_location=device)

# Recover class names
if class_names_override:
    class_names: List[str] = list(class_names_override)
else:
    class_to_idx: dict = ckpt['class_to_idx']
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

num_classes = len(class_names)
print(f'Classes ({num_classes}): {class_names}')

model = FusionClassifier(
    text_model_name=text_model_name,
    num_classes=num_classes,
    dropout=dropout,
    image_encoder_name=image_encoder_name,
    text_encoder_name=text_encoder_name,
).to(device)

model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('Checkpoint loaded successfully.')

## 3. Build Test DataLoader

In [ ]:
def _import_multimodal_ingestion(src_dir: Path):
    module_path = src_dir / 'data_ingestion' / 'multimodal_ingestion.py'
    spec = importlib.util.spec_from_file_location('multimodal_ingestion', module_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

ingestion = _import_multimodal_ingestion(src_dir)

# We only need the test loader; supply dummy train/val paths equal to test
_, _, test_loader, _ = ingestion.make_loaders(
    train_path=test_path,   # unused but required
    val_path=test_path,     # unused but required
    test_path=test_path,
    batch_size=batch_size,
    num_workers=num_workers,
    tokenizer_name=text_model_name,
    max_length=max_length,
    local_files_only=local_files_only,
)
print(f'Test batches: {len(test_loader)}')

## 4. Collect Predictions

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    paths, sents = [], []
    have_path = have_sent = True

    for batch in loader:
        pixel_values  = batch['pixel_values'].to(device)
        input_ids     = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels        = batch['label'].to(device)

        logits = model(pixel_values, input_ids, attention_mask)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())
        y_prob.append(probs.cpu().numpy())

        if have_path:
            if 'path' in batch: paths.extend([str(p) for p in batch['path']])
            else: have_path = False
        if have_sent:
            if 'sentence' in batch: sents.extend([str(t) for t in batch['sentence']])
            else: have_sent = False

    y_true_np = np.array(y_true, dtype=int)
    y_pred_np = np.array(y_pred, dtype=int)
    y_prob_np = np.concatenate(y_prob, axis=0)
    return y_true_np, y_pred_np, y_prob_np, (paths if have_path else None), (sents if have_sent else None)


y_true, y_pred, y_prob, paths, sentences = collect_predictions(model, test_loader, device)
conf = y_prob.max(axis=1)
correct = (y_pred == y_true)

print(f'Samples : {len(y_true)}')
print(f'Accuracy: {correct.mean():.4f}')

## 5. Classification Report

In [ ]:
report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report_str)

## 6. Per-Class Metrics Table

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(num_classes)), zero_division=0
)

metrics_df = pd.DataFrame({
    'Class':     class_names,
    'Precision': precision.round(4),
    'Recall':    recall.round(4),
    'F1':        f1.round(4),
    'Support':   support.astype(int),
})

display(metrics_df.style
    .background_gradient(subset=['Precision','Recall','F1'], cmap='YlGn')
    .format({'Precision':'{:.4f}','Recall':'{:.4f}','F1':'{:.4f}'})
    .set_caption('Per-Class Metrics')
)

## 7. Confusion Matrices

In [ ]:
def plot_cm(cm, class_names, normalize=False, title=None):
    mat = cm.astype(float)
    if normalize:
        row_sums = mat.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        mat = mat / row_sums

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(mat, cmap='Blues')
    fig.colorbar(im, ax=ax)
    ax.set_title(title or ('Confusion Matrix' + (' (Normalized)' if normalize else ' (Counts)')))
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            txt = f'{mat[i,j]*100:.1f}%' if normalize else str(int(cm[i,j]))
            color = 'white' if mat[i,j] > mat.max()*0.6 else 'black'
            ax.text(j, i, txt, ha='center', va='center', color=color, fontsize=9)
    plt.tight_layout()
    plt.show()


cm = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[int(t), int(p)] += 1

plot_cm(cm, class_names, normalize=False, title='Confusion Matrix — Counts')
plot_cm(cm, class_names, normalize=True,  title='Confusion Matrix — Normalized')

## 8. ROC Curves (One-vs-Rest)

In [ ]:
y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))

fig, ax = plt.subplots(figsize=(8, 6))
for c in range(num_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, c], y_prob[:, c])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{class_names[c]} (AUC={roc_auc:.3f})')
ax.plot([0,1],[0,1],'--', color='grey')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (One-vs-Rest) — Test')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 9. Precision-Recall Curves (One-vs-Rest)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for c in range(num_classes):
    prec, rec, _ = precision_recall_curve(y_true_bin[:, c], y_prob[:, c])
    ap = average_precision_score(y_true_bin[:, c], y_prob[:, c])
    ax.plot(rec, prec, label=f'{class_names[c]} (AP={ap:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (One-vs-Rest) — Test')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 10. Calibration

In [ ]:
n_bins = 10
bins   = np.linspace(0.0, 1.0, n_bins + 1)
bin_ids = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)

bin_conf  = np.zeros(n_bins)
bin_acc   = np.zeros(n_bins)
bin_count = np.zeros(n_bins, dtype=int)

for b in range(n_bins):
    m = bin_ids == b
    bin_count[b] = int(m.sum())
    if bin_count[b] > 0:
        bin_conf[b] = conf[m].mean()
        bin_acc[b]  = correct.astype(float)[m].mean()

# Calibration curve
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0,1],[0,1],'--', color='grey', label='Perfect calibration')
ax.plot(bin_conf, bin_acc, marker='o', label='Model')
ax.set_xlabel('Mean confidence (max prob)')
ax.set_ylabel('Accuracy')
ax.set_title('Calibration Curve — Test')
ax.legend()
plt.tight_layout()
plt.show()

# Confidence histogram
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(conf, bins=20, edgecolor='white')
ax.set_xlabel('Predicted confidence (max prob)')
ax.set_ylabel('Count')
ax.set_title('Confidence Histogram — Test')
plt.tight_layout()
plt.show()

# Accuracy vs confidence bin
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(n_bins)
ax.bar(x, bin_acc)
ax.set_xticks(x)
ax.set_xticklabels([f'{bins[i]:.1f}-{bins[i+1]:.1f}' for i in range(n_bins)], rotation=45, ha='right')
ax.set_ylim(0, 1)
ax.set_xlabel('Confidence bin')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Confidence Bin — Test')
plt.tight_layout()
plt.show()

## 11. Top Confident Wrong Predictions

In [ ]:
TOP_K = 50

wrong = np.where(y_true != y_pred)[0]
sorted_wrong = wrong[np.argsort(-conf[wrong])][:TOP_K] if wrong.size else np.array([], dtype=int)

rows = []
for rank, i in enumerate(sorted_wrong, start=1):
    rows.append({
        'Rank':       rank,
        'Confidence': round(float(conf[i]), 4),
        'True':       class_names[int(y_true[i])],
        'Predicted':  class_names[int(y_pred[i])],
        'Path':       (paths[i] if paths else ''),
        'Sentence':   (sentences[i] if sentences else ''),
    })

wrong_df = pd.DataFrame(rows)
display(wrong_df.style
    .background_gradient(subset=['Confidence'], cmap='Reds')
    .set_caption(f'Top-{TOP_K} Most Confident Wrong Predictions')
)

## 12. Misclassified Image Grid *(requires `path` and `sentence` in dataset)*

In [ ]:
import importlib
PIL_spec = importlib.util.find_spec('PIL')

if paths is None:
    print('Skipping grid — dataset does not return "path" field.')
elif wrong.size == 0:
    print('No misclassified samples — model is perfect on this set!')
elif PIL_spec is None:
    print('Skipping grid — Pillow not installed.')
else:
    from PIL import Image as PILImage

    MAX_ITEMS = 25
    pick = wrong[np.argsort(-conf[wrong])][:MAX_ITEMS]
    n = len(pick)
    cols = 5
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.2))
    axes = np.array(axes).flatten()

    for k, i in enumerate(pick):
        img = PILImage.open(paths[i]).convert('RGB')
        axes[k].imshow(img)
        axes[k].axis('off')
        t   = class_names[int(y_true[i])]
        p   = class_names[int(y_pred[i])]
        c   = float(conf[i])
        sent = (sentences[i][:40] if sentences else '')
        title = f'T:{t}  P:{p} ({c:.2f})'
        if sent: title += '\n' + sent
        axes[k].set_title(title, fontsize=8)

    for k in range(n, len(axes)):
        axes[k].axis('off')

    fig.suptitle('Top Misclassified Samples (highest confidence)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()